# पुनरावर्ती न्यूरल नेटवर्क

पिछले मॉड्यूल में, हमने टेक्स्ट के समृद्ध सेमांटिक प्रतिनिधित्व को कवर किया था। वास्तुकला जिसका हम उपयोग कर रहे हैं, वाक्य में शब्दों के समेकित अर्थ को कैप्चर करती है, लेकिन यह शब्दों के **क्रम** को ध्यान में नहीं रखती क्योंकि एम्बेडिंग के बाद जो समेकन ऑपरेशन होता है वह मूल टेक्स्ट से यह जानकारी हटा देता है। क्योंकि ये मॉडल शब्दों के क्रम को प्रस्तुत करने में असमर्थ हैं, वे अधिक जटिल या अस्पष्ट कार्यों जैसे टेक्स्ट जनरेशन या प्रश्नोत्तर को हल नहीं कर सकते।

एक टेक्स्ट अनुक्रम के अर्थ को पकड़ने के लिए, हम एक न्यूरल नेटवर्क वास्तुकला का उपयोग करेंगे जिसे **पुनरावर्ती न्यूरल नेटवर्क** या RNN कहा जाता है। RNN का उपयोग करते समय, हम अपना वाक्य एक समय में एक टोकन के माध्यम से नेटवर्क को पास करते हैं, और नेटवर्क कुछ **स्थिति** उत्पन्न करता है, जिसे हम फिर अगले टोकन के साथ नेटवर्क को फिर से पास करते हैं।

![Image showing an example recurrent neural network generation.](../../../../../translated_images/hi/rnn.27f5c29c53d727b5.webp)

टोकन के इनपुट अनुक्रम $X_0,\dots,X_n$ के लिए, RNN न्यूरल नेटवर्क ब्लॉकों का एक अनुक्रम बनाता है, और इस अनुक्रम को बैकप्रोपेगेशन का उपयोग करके एंड-टू-एंड प्रशिक्षित करता है। प्रत्येक नेटवर्क ब्लॉक इनपुट के रूप में एक जोड़ी $(X_i,S_i)$ लेता है, और परिणामस्वरूप $S_{i+1}$ उत्पन्न करता है। अंतिम स्थिति $S_n$ या आउटपुट $Y_n$ परिणाम उत्पन्न करने के लिए एक रैखिक क्लासिफायर में जाता है। सभी नेटवर्क ब्लॉक एक ही भार साझा करते हैं, और एक बैकप्रोपेगेशन पास का उपयोग करके एंड-टू-एंड प्रशिक्षित होते हैं।

> ऊपर की आकृति में पुनरावर्ती न्यूरल नेटवर्क को अनरोल्ड रूप (बाएँ) में और अधिक संकुचित पुनरावर्ती प्रतिनिधित्व (दाएँ) में दिखाया गया है। यह समझना महत्वपूर्ण है कि सभी RNN कोशिकाओं के पास समान **साझा करने योग्य भार** होते हैं।

क्योंकि स्थिति वेक्टर $S_0,\dots,S_n$ नेटवर्क के माध्यम से भेजे जाते हैं, RNN शब्दों के बीच क्रमिक निर्भरताओं को सीखने में सक्षम है। उदाहरण के लिए, जब शब्द *not* अनुक्रम में कहीं आता है, तो यह स्थिति वेक्टर के कुछ तत्वों को नकारना सीख सकता है।

अंदर, प्रत्येक RNN सेल में दो भार मैट्रिक्स होते हैं: $W_H$ और $W_I$, और बायस $b$। प्रत्येक RNN चरण में, इनपुट $X_i$ और इनपुट स्थिति $S_i$ के दिए जाने पर, आउटपुट स्थिति की गणना $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ के रूप में होती है, जहाँ $f$ एक सक्रियता फ़ंक्शन है (अक्सर $\tanh$)।

> टेक्स्ट जनरेशन जैसे समस्याओं के लिए (जिसे हम अगले यूनिट में कवर करेंगे) या मशीन ट्रांसलेशन के लिए हम प्रत्येक RNN चरण पर कुछ आउटपुट मान भी प्राप्त करना चाहते हैं। इस मामले में, एक अन्य मैट्रिक्स $W_O$ भी होता है, और आउटपुट की गणना $Y_i=f(W_O\times S_i+b_O)$ के रूप में की जाती है।

आइए देखें कि पुनरावर्ती न्यूरल नेटवर्क हमारे न्यूज डेटासेट को वर्गीकृत करने में कैसे मदद कर सकते हैं।

> सैंडबॉक्स वातावरण के लिए, हमें यह सुनिश्चित करने के लिए निम्न सेल को चलाना होगा कि आवश्यक पुस्तकालय स्थापित है, और डेटा प्रीफ़ेच किया गया है। यदि आप लोकली चला रहे हैं, तो आप निम्न सेल को छोड़ सकते हैं।


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

बड़े मॉडल प्रशिक्षण के दौरान, GPU मेमोरी आवंटन समस्या बन सकता है। हमें विभिन्न मिनीबैच आकारों के साथ प्रयोग भी करना पड़ सकता है, ताकि डेटा हमारे GPU मेमोरी में फिट हो जाए, और प्रशिक्षण पर्याप्त तेज़ भी हो। यदि आप इस कोड को अपनी खुद की GPU मशीन पर चला रहे हैं, तो आप प्रशिक्षण को तेज़ करने के लिए मिनीबैच आकार को समायोजित करने का प्रयोग कर सकते हैं।

> **Note**: NVidia ड्राइवर के कुछ संस्करणों के बारे में जाना गया है कि वे मॉडल प्रशिक्षण के बाद मेमोरी को रिलीज़ नहीं करते हैं। हम इस नोटबुक में कई उदाहरण चला रहे हैं, और यह कुछ सेटअप्स में मेमोरी समाप्त होने का कारण बन सकता है, खासकर यदि आप उसी नोटबुक के हिस्से के रूप में अपने स्वयं के प्रयोग कर रहे हैं। यदि आपको मॉडल प्रशिक्षण शुरू करते समय कुछ अजीब त्रुटियाँ मिलती हैं, तो आप नोटबुक कर्नेल को पुनः प्रारंभ कर सकते हैं।


In [3]:
batch_size = 16
embed_size = 64

## सरल RNN वर्गीकारक

एक सरल RNN के मामले में, प्रत्येक पुनरावर्ती इकाई एक सरल रैखिक नेटवर्क होती है, जो एक इनपुट वेक्टर और स्थिति वेक्टर लेती है, और एक नया स्थिति वेक्टर उत्पन्न करती है। Keras में, इसे `SimpleRNN` परत द्वारा प्रस्तुत किया जा सकता है।

जबकि हम RNN परत को सीधे वन-हॉट एनकोडेड टोकन दे सकते हैं, यह उनकी उच्च आयामिता के कारण एक अच्छा विचार नहीं है। इसलिए, हम शब्द वेक्टरों की आयामिता कम करने के लिए एक एम्बेडिंग परत का उपयोग करेंगे, उसके बाद एक RNN परत, और अंत में एक `Dense` वर्गीकारक।

> **नोट**: उन मामलों में जहाँ आयामिता इतनी अधिक नहीं होती, उदाहरण के लिए चरित्र-स्तरीय टोकनाइजेशन का उपयोग करते समय, वन-हॉट एनकोडेड टोकन को सीधे RNN सेल में पास करना समझदारी हो सकती है।


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **नोट:** यहाँ हम सरलता के लिए एक अनट्रेंड एम्बेडिंग लेयर का उपयोग करते हैं, लेकिन बेहतर परिणामों के लिए हम पूर्व-प्रशिक्षित एम्बेडिंग लेयर का उपयोग कर सकते हैं जैसे कि पिछले यूनिट में Word2Vec का वर्णन किया गया था। यह आपके लिए एक अच्छा अभ्यास होगा कि आप इस कोड को पूर्व-प्रशिक्षित एम्बेडिंग के साथ काम करने के लिए अनुकूलित करें।

अब चलिए अपने RNN को प्रशिक्षित करते हैं। सामान्यतः RNN को प्रशिक्षित करना काफी कठिन होता है, क्योंकि जब RNN सेल्स को सीक्वेंस की लंबाई के अनुसार अनरोल किया जाता है, तो बैकप्रोपेगेशन में शामिल लेयर्स की संख्या काफी बड़ी हो जाती है। इसलिए हमें एक छोटा लर्निंग रेट चुनना पड़ता है, और अच्छे परिणाम प्राप्त करने के लिए नेटवर्क को बड़े डेटासेट पर प्रशिक्षित करना पड़ता है। यह काफी समय ले सकता है, इसलिए GPU का उपयोग करना बेहतर होता है।

समय बचाने के लिए, हम केवल न्यूज शीर्षकों पर RNN मॉडल को प्रशिक्षित करेंगे, विवरण को छोड़कर। आप विवरण के साथ प्रशिक्षण करने की कोशिश कर सकते हैं और देख सकते हैं कि क्या आप मॉडल को प्रशिक्षित कर पाते हैं।


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **ध्यान दें** कि सटीकता यहां कम हो सकती है, क्योंकि हम केवल समाचार शीर्षकों पर ही प्रशिक्षण दे रहे हैं।


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


अब जब हम मास्किंग का उपयोग कर रहे हैं, तो हम पूरी शीर्षक और विवरण के डेटासेट पर मॉडल को प्रशिक्षित कर सकते हैं।

> **टिप्पणी**: क्या आपने ध्यान दिया है कि हम समाचार शीर्षकों पर प्रशिक्षित एक वेक्टराइज़र का उपयोग कर रहे हैं, न कि लेख के पूरे भाग पर? संभवतः, इससे कुछ टोकन को नजरअंदाज किया जा सकता है, इसलिए वेक्टराइज़र को पुनः प्रशिक्षित करना बेहतर होगा। हालांकि, इसका प्रभाव बहुत छोटा हो सकता है, इसलिए हम सरलता के लिए पहले से प्रशिक्षित वेक्टराइज़र पर ही टिके रहेंगे।


## LSTM: लंबी छोटी अवधि की मेमोरी

RNNs की मुख्य समस्याओं में से एक है **ग्रेडिएंट्स का घट जाना**। RNNs काफी लंबे हो सकते हैं, और बैकप्रोपगेशन के दौरान ग्रेडिएंट्स को नेटवर्क की पहली परत तक वापस फैलाने में कठिनाई हो सकती है। जब ऐसा होता है, तो नेटवर्क दूर के टोकनों के बीच संबंध सीख नहीं पाता। इस समस्या से बचने का एक तरीका है **गेट्स** का उपयोग करके **स्पष्ट राज्य प्रबंधन** प्रस्तुत करना। सबसे सामान्य वास्तुकलाएं जो गेट्स पेश करती हैं वह हैं **लंबी छोटी अवधि की मेमोरी** (LSTM) और **गेटेड रिले यूनिट** (GRU)। हम यहाँ LSTM को कवर करेंगे।

![लंबी छोटी अवधि की मेमोरी सेल का एक उदाहरण दिखाता हुआ चित्र](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

एक LSTM नेटवर्क RNN के समान तरीके से व्यवस्थित होता है, लेकिन दो राज्य होते हैं जो परत दर परत पास होते हैं: वास्तविक राज्य $c$, और छुपा वेक्टर $h$. प्रत्येक यूनिट में, छुपा वेक्टर $h_{t-1}$ को इनपुट $x_t$ के साथ मिलाया जाता है, और वे मिलकर **गेट्स** के माध्यम से यह नियंत्रित करते हैं कि राज्य $c_t$ और आउटपुट $h_{t}$ के साथ क्या होगा। प्रत्येक गेट में सिग्मॉइड सक्रियण होता है (आउटपुट सीमा $[0,1]$ में), जिसे राज्य वेक्टर के साथ गुणा करने पर बिटवाइज मास्क की तरह माना जा सकता है। LSTM में निम्नलिखित गेट्स होते हैं (ऊपर चित्र में बाईं से दाईं ओर):
* **फॉरगेट गेट** जो यह निर्धारित करता है कि वेक्टर $c_{t-1}$ के कौन से घटक भूलना है, और कौन से पास करना है।
* **इनपुट गेट** जो यह तय करता है कि इनपुट वेक्टर और पिछले छुपे वेक्टर से कितनी जानकारी राज्य वेक्टर में शामिल की जानी चाहिए।
* **आउटपुट गेट** जो नए राज्य वेक्टर को लेता है और निर्णय करता है कि इसके कौन से घटक नए छुपे वेक्टर $h_t$ को बनाने के लिए उपयोग किए जाएंगे।

राज्य $c$ के घटकों को उन झंडों के रूप में सोचा जा सकता है जिन्हें चालू और बंद किया जा सकता है। उदाहरण के लिए, जब हमें अनुक्रम में *Alice* नाम मिलता है, तो हम अनुमान लगाते हैं कि यह एक महिला को संदर्भित करता है, और राज्य में वह झंडा उठाते हैं जो कहता है कि वाक्य में एक स्त्रीलिंग संज्ञा है। जब हमें आगे शब्द *and Tom* मिलते हैं, तो हम वह झंडा उठाएंगे जो कहता है कि एक बहुवचन संज्ञा है। इस प्रकार, राज्य को नियंत्रित करके हम वाक्य के व्याकरणिक गुणों का पता रख सकते हैं।

> **टिप्पणी**: LSTM के आंतरिक संरचना को समझने के लिए यह एक शानदार संसाधन है: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) क्रिस्टोफर ओलाह द्वारा।

जबकि एक LSTM सेल की आंतरिक संरचना जटिल लग सकती है, केरस इसे `LSTM` परत के अंदर छिपा देता है, इसलिए ऊपर के उदाहरण में करने के लिए केवल इतना है कि हम पुनरावर्ती परत को बदल दें:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **ध्यान दें** कि LSTM को ट्रेन करना भी बहुत धीमा होता है, और आपको प्रशिक्षण की शुरुआत में सटीकता में ज्यादा वृद्धि नहीं दिख सकती है। अच्छी सटीकता प्राप्त करने के लिए आपको कुछ समय तक प्रशिक्षण जारी रखना पड़ सकता है।


## द्विदिशात्मक और बहु-परत RNNs

अब तक हमारे उदाहरणों में, पुनरावर्ती नेटवर्क अनुक्रम की शुरुआत से अंत तक काम करते हैं। यह हमें प्राकृतिक लगता है क्योंकि यह उसी दिशा का अनुसरण करता है जिसमें हम पढ़ते या बोलते हुए सुनते हैं। हालाँकि, उन स्थितियों के लिए जहाँ इनपुट अनुक्रम के यादृच्छिक पहुँच की आवश्यकता होती है, पुनरावर्ती गणना दोनों दिशाओं में चलाना अधिक अर्थपूर्ण होता है। ऐसे RNNs जो दोनों दिशाओं में गणनाओं की अनुमति देते हैं, उन्हें **द्विदिशात्मक** RNNs कहा जाता है, और इन्हें खास `Bidirectonal` लेयर के साथ पुनरावर्ती लेयर को लपेट कर बनाया जा सकता है।

> **नोट**: `Bidirectional` लेयर इसके भीतर लेयर की दो प्रतियां बनाता है, और उन प्रतियों में से एक की `go_backwards` प्रॉपर्टी को `True` पर सेट करता है, जिससे यह अनुक्रम के विपरीत दिशा में चलता है।

पुनरावर्ती नेटवर्क, एकदिशात्मक या द्विदिशात्मक, अनुक्रम के भीतर पैटर्न पकड़ते हैं, और उन्हें राज्य वेक्टर में संग्रहित करते हैं या आउटपुट के रूप में लौटाते हैं। कन्वोल्यूशनल नेटवर्क की तरह, हम पहले के बाद एक और पुनरावर्ती लेयर बना सकते हैं ताकि उच्च स्तरीय पैटर्न पकड़े जा सकें, जो पहली लेयर द्वारा निकाले गए निम्न स्तरीय पैटर्न से बने होते हैं। यह हमें **मल्टी-लेयर RNN** के विचार की ओर ले जाता है, जिसमें दो या अधिक पुनरावर्ती नेटवर्क होते हैं, जहाँ पिछले लेयर का आउटपुट अगले लेयर को इनपुट के रूप में दिया जाता है।

![मल्टीलेयर लॉन्ग-शॉर्ट-टर्म मेमोरी RNN दिखाने वाला चित्र](../../../../../translated_images/hi/multi-layer-lstm.dd975e29bb2a59fe.webp)

*तस्वीर [इस शानदार पोस्ट](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) से है, फर्नांडो लोपेज द्वारा।*

केरस इन्हें बनाना आसान बनाता है, क्योंकि आपको मॉडल में बस और पुनरावर्ती लेयर जोड़नी होती हैं। अंतिम लेयर को छोड़कर सभी लेयर्स के लिए, हमें `return_sequences=True` पैरामीटर निर्दिष्ट करना होता है, क्योंकि हमें लेयर को सभी मध्यवर्ती राज्य लौटाने होते हैं, न कि केवल पुनरावर्ती गणना की अंतिम स्थिति।

आइए हमारे वर्गीकरण समस्या के लिए एक द्विदिशात्मक LSTM के दो-लेयर बनाएं।

> **नोट** यह कोड भी पूरा होने में काफी समय लेता है, लेकिन यह हमें अब तक की सबसे अधिक सटीकता देता है। इसलिए शायद परिणाम देखने के लिए इंतजार करना उचित है।


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## अन्य कार्यों के लिए RNNs

अब तक, हमने RNNs का उपयोग करके टेक्स्ट के अनुक्रमों को वर्गीकृत करने पर ध्यान केंद्रित किया है। लेकिन वे कई अन्य कार्यों को भी संभाल सकते हैं, जैसे कि टेक्स्ट जनरेशन और मशीन अनुवाद &mdash; हम इन कार्यों पर अगले इकाई में विचार करेंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
